# 專題A_智慧視覺打卡系統/起手式/打卡系統.py
**電腦視覺與深度學習 › Day16 期中專題整合日**

期中專題 A 的起手式:開鏡頭、按空白鍵打卡、資料存進 SQLite、按 c 看統計圖。它把影像、資料庫和圖表三天的技能接在一起。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/2/0)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY8/專題A_智慧視覺打卡系統/起手式"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '打卡系統.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 說明:現在會做什麼、要你加什麼
開頭的註解列出起手式已經能做的四件事,以及你要接手補的功能,例如影格差異自動打卡、遲到判斷。先跑起來,再一項一項加。

In [ ]:
# 專題 A 起手式：智慧視覺打卡系統（能跑的最小骨架）

### 匯入與資料庫路徑
sqlite3 是 Python 內建的資料庫,datetime 拿來記時間。資料庫檔放在這支程式旁邊,叫 attendance.db。

In [ ]:
#
# 現在會做的事：
#   1. 開鏡頭即時顯示
#   2. 按 SPACE 幫「目前姓名」打一次卡，寫進 SQLite
#   3. 按 c 跳出 matplotlib 出勤統計圖
#   4. 按 n 重新輸入姓名（在終端機打）、按 q 離開
#
# 你要接手加的（README 的必做 / 加分，程式裡用 TODO 標出來）：
#   - 影格差異自動打卡、tkinter GUI 包起來、遲到判斷、匯出 Excel …

import sqlite3
from datetime import datetime
from pathlib import Path

import cv2

BASE = Path(__file__).parent

### 建立資料表
第一次執行時建立「打卡紀錄」這張表,有編號、姓名、時間三個欄位。IF NOT EXISTS 讓第二次執行不會重複建立。

In [ ]:
DB_PATH = BASE / "attendance.db"


# ====== 資料庫層 ======
def 初始化資料庫():
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        """CREATE TABLE IF NOT EXISTS 打卡紀錄 (
               id INTEGER PRIMARY KEY AUTOINCREMENT,
               姓名 TEXT NOT NULL,
               時間 TEXT NOT NULL
           )"""
    )
    conn.commit()
    return conn

### 新增一筆打卡
把現在的時間格式化成年月日時分秒,用 INSERT 寫進資料表。問號是參數佔位符,可以避免 SQL 注入。最後 commit 才算真正存進去。

In [ ]:
def 新增打卡(conn, 姓名):
    現在 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn.execute("INSERT INTO 打卡紀錄 (姓名, 時間) VALUES (?, ?)", (姓名, 現在))
    conn.commit()
    print(f"[打卡] {姓名} @ {現在}")

### 每個人打了幾次卡
GROUP BY 姓名把同一個人的紀錄分成一組,COUNT 數每組有幾筆,再由多到少排序。這就是 Day13 練過的 SQL。

In [ ]:
def 統計每人次數(conn):
    cur = conn.execute(
        "SELECT 姓名, COUNT(*) FROM 打卡紀錄 GROUP BY 姓名 ORDER BY COUNT(*) DESC"
    )
    return cur.fetchall()  # [(姓名, 次數), ...]

### 畫出勤長條圖
沒資料就提示一下。有資料就用 matplotlib 畫長條圖,每個人一根柱子。先設定微軟正黑體,中文才不會變方框。

In [ ]:
# ====== 統計圖 ======
def 顯示統計圖(conn):
    資料 = 統計每人次數(conn)
    if not 資料:
        print("目前還沒有任何打卡紀錄。")
        return

    import matplotlib.pyplot as plt

    plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei"]  # 中文不變方框
    plt.rcParams["axes.unicode_minus"] = False

    姓名清單 = [r[0] for r in 資料]
    次數清單 = [r[1] for r in 資料]

    plt.figure("出勤統計")
    plt.bar(姓名清單, 次數清單, color="#4C72B0")
    plt.title("每人打卡次數")
    plt.ylabel("次數")
    plt.tight_layout()
    plt.show()

In [ ]:
    # TODO(加分)：改成出勤/缺席圓餅圖、遲到用不同顏色

### 輸入姓名、開鏡頭
先建好資料庫,在終端機輸入姓名,沒輸入就叫訪客。接著打開鏡頭;打不開的話,也可以先不接鏡頭,單獨測資料庫函式。

### 畫面上疊文字
每一格畫面寫上目前的姓名和操作說明。注意 OpenCV 的 putText 不支援中文,所以畫面上都寫英文,中文留給圖表或 GUI。

### 按鍵控制
這裡標了一個加分 TODO:可以加影格差異偵測,有人變化就自動打卡。下面讀鍵盤:空白鍵打卡、c 看圖、n 改姓名、q 離開。

### 收拾與進入點
離開迴圈後放掉鏡頭、關視窗、關資料庫連線。最後兩行讓這支檔案直接執行時才跑 main。

In [ ]:
# ====== 主程式 ======
def main():
    conn = 初始化資料庫()

    姓名 = input("請輸入你的姓名：").strip() or "訪客"

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("錯誤：無法開啟鏡頭。可先不接鏡頭，改用終端機測試資料庫函式。")
        return

    print("SPACE=打卡  c=看統計圖  n=改姓名  q=離開")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 畫面上疊資訊（cv2.putText 只吃英文，中文請用 matplotlib/GUI 呈現）
        cv2.putText(frame, f"Name: {姓名}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(frame, "SPACE=punch  c=chart  n=name  q=quit", (10, 460),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        # TODO(加分)：這裡加「影格差異」偵測，畫面有人變化就自動打卡
        cv2.imshow("attendance", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord(" "):
            新增打卡(conn, 姓名)
        elif key == ord("c"):
            顯示統計圖(conn)
        elif key == ord("n"):
            姓名 = input("新姓名：").strip() or 姓名

    cap.release()
    cv2.destroyAllWindows()
    conn.close()


if __name__ == "__main__":
    main()

## 小結
起手式已經把鏡頭、資料庫、圖表接通了,你接下來的工作是一項一項補 TODO。記得每補一項就跑一次。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/2/0) 看逐句拆解、做練習。